# KilnWatch — main training run (yolo11l-obb)

Meant for **Save Version → Save & Run All (Commit)**, so it runs in the background and survives a closed browser or a sleeping laptop.

Settings before saving: dataset `sentinelkiln-dataset` (by rishabhsnip) added as input, **Accelerator: GPU T4 x2** (or P100), **Internet: on**.

Training stops after 9 hours, with the learning-rate schedule fitted to that budget, leaving room for data prep and final scoring inside Kaggle's 12-hour limit. Results land in this version's **Output** tab under `runs/kilnwatch/main/`: `weights/best.pt` and `scores.json`.

In [ ]:
!pip install -q ultralytics
!git clone -q https://github.com/aryamanraj2/KilnWatch.git

In [ ]:
from pathlib import Path
SRC = next(p for pat in ("*/train/images", "*/*/train/images", "*/*/*/train/images")
           for p in Path("/kaggle/input").glob(pat)).parents[1]
print(SRC)
!python KilnWatch/scripts/prepare_data.py --src "{SRC}" --dst /tmp/kilns --link

In [ ]:
!python KilnWatch/scripts/train.py --data /tmp/kilns/kilns_full.yaml --model yolo11l-obb.pt --epochs 100 --hours 9 --batch 64 --cache ram --name main --project /kaggle/working/runs/kilnwatch

In [ ]:
import json
s = json.load(open("/kaggle/working/runs/kilnwatch/main/scores.json"))
print("epochs run:", s.get("epochs_run"), "| minutes:", s["train_minutes"])
print(json.dumps(s["test"], indent=2))